# Tabular RAG — Text-to-Pandas with Self-Correction
## Tables as Context — NL Queries on Structured Data
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/58-tabular-rag/tabular_rag_workbook.ipynb)

Standard embedding-based RAG breaks on structured data: a question like "which product has the highest revenue?" doesn't find the right row via cosine similarity — it needs computation. **Tabular RAG** replaces retrieval with code generation: the LLM writes a pandas expression, the system executes it, and if it fails, the error is fed back to the LLM for self-correction. By the end of this workshop you will understand *why* tabular data needs a different strategy, *how* the text-to-pandas loop works, and *how* to wire the self-correction cycle into a LangGraph conditional edge.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why embedding-based RAG fails on tables |
| 2 | **Schema Inspection** — What the LLM sees before generating code |
| 3 | **Text-to-Pandas** — LLM writes an expression, we eval() it |
| 4 | **Self-Correction Loop** — Error feedback → retry |
| 5 | **Full Pipeline** — LangGraph with conditional retry edge |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with `.venv` and `requirements.txt` installed, **or** Google Colab (install cell below)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Chen, W. et al. (2020). *Open Domain Question Answering over Tables via Dense Retrieval.* https://arxiv.org/abs/2010.12049  
> Gao, S. et al. (2023). *Evaluating LLMs on Tabular Data Tasks.* https://arxiv.org/abs/2305.13062

In [ ]:
# Install dependencies — runs only on Google Colab.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "langchain",
        "langchain-openai",
        "langgraph",
        "pandas",
        "python-dotenv",
    ])
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running LLM cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — Why Embedding-Based RAG Fails on Tables

---

### The Problem With Chunking Tables

Imagine you chunk a CSV into one chunk per row:

```
Chunk 0: "Laptop Pro | Electronics | 1299.99 | 450 | North"
Chunk 1: "Wireless Mouse | Electronics | 29.99 | 2100 | North"
...
Chunk 9: "Desk Lamp | Furniture | 49.99 | 1450 | South"
```

Query: "Which product has the highest revenue?"

Revenue = price × units_sold. No single row tells you that — you'd need to *compute* it across all rows and find the max. The top-k retrieved chunks might not even include the right answer. And even if you passed all 10 chunks to the LLM, it would have to do arithmetic in its head (unreliable).

**Tabular RAG** takes a completely different approach:

```
QUESTION: "Which product has the highest total revenue?"
       │
       ▼
LLM GENERATES CODE:
  df.assign(revenue=df.price * df.units_sold).loc[lambda x: x.revenue.idxmax(), 'product']
       │
EXECUTE:  eval() → "Notebook"
       │
       ▼
ANSWER: "Notebook"
```

The LLM never looks at the data — it writes code that looks at the data. This scales to millions of rows and handles aggregations, filtering, and math that would overwhelm in-context reasoning.

### When to Use Tabular RAG vs Embedding RAG

| Question type | Approach |
|---------------|----------|
| "What is the policy on returns?" (prose doc) | Embedding RAG |
| "Which region sold the most units?" (aggregation) | Tabular RAG |
| "Summarise the Q3 earnings report" (long text) | Embedding RAG |
| "What is the average price for Electronics?" (computation) | Tabular RAG |
| "What does Einstein's Nobel Prize mean historically?" | Embedding RAG |

In [ ]:
import io

import pandas as pd

SAMPLE_CSV = """\
product,category,price,units_sold,region
Laptop Pro,Electronics,1299.99,450,North
Wireless Mouse,Electronics,29.99,2100,North
Office Chair,Furniture,349.99,820,South
Standing Desk,Furniture,599.99,340,South
Notebook,Stationery,4.99,5500,East
Pen Pack,Stationery,9.99,3200,East
Coffee Maker,Appliances,89.99,670,West
Blender Pro,Appliances,149.99,290,West
Monitor 27in,Electronics,399.99,610,North
Desk Lamp,Furniture,49.99,1450,South
"""

SAMPLE_QUESTIONS = [
    "Which product has the highest total revenue (price * units_sold)?",
    "What is the average price of Electronics products?",
    "How many units were sold in the North region in total?",
]

df = pd.read_csv(io.StringIO(SAMPLE_CSV))
print(f"Table shape: {df.shape}")
print(df.to_string(index=False))

---

## Part 2 — Schema Inspection

---

Before asking the LLM to generate code, we give it a **schema summary** (column names, dtypes) and a **sample of rows**. This is the "context" the LLM gets instead of embedded chunks. The richer this schema prompt, the better the generated code.

In [ ]:
# ─── Build the schema context the LLM will receive ───────────────────────────

# Schema + a few sample rows act as the LLM's "table of contents" —
# without dtypes the model guesses; without sample rows it can't see actual values.
schema = f"columns: {list(df.columns)}, dtypes: {df.dtypes.to_dict()}"
sample = df.head(3).to_string(index=False)

print("Schema:")
print(f"  {schema}")
print()
print("Sample rows (first 3):")
for line in sample.split("\n"):
    print(f"  {line}")

In [ ]:
# ─── The query prompt template ────────────────────────────────────────────────

# Constraining to a single expression (no assignment, no print) lets eval() work
# on the raw output; asking for a scalar avoids unprintable DataFrame returns.
QUERY_PROMPT = """\
You have a pandas DataFrame named `df` with these columns: {columns}
Sample rows:
{sample}

Write a single Python expression (no print, no assignment) that answers:
{question}

The expression must evaluate to a scalar value or a short string.
Return only the expression, nothing else."""

# Preview the prompt for question 0
print(QUERY_PROMPT.format(
    columns=list(df.columns),
    sample=sample,
    question=SAMPLE_QUESTIONS[0],
))

---

## Part 3 — Text-to-Pandas (Single Shot)

---

Ask the LLM to write a pandas expression for each question, then evaluate it.

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

# temperature=0 → deterministic code generation; higher values can produce
# creative but inconsistent pandas expressions that fail eval().
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def text_to_pandas(question: str, error: str = "") -> str:
    prompt = QUERY_PROMPT.format(
        columns=list(df.columns),
        sample=sample,
        question=question,
    )
    if error:
        prompt += f"\n\nPrevious expression raised: {error}\nFix it."
    return llm.invoke([HumanMessage(content=prompt)]).content.strip()


print(f"{'Question':<55} {'Generated Expression':<70} Result")
print("-" * 140)

for q in SAMPLE_QUESTIONS:
    code = text_to_pandas(q)
    try:
# Isolated namespace {"df": df, "pd": pd} prevents the generated code from
# touching globals — a lightweight sandbox for untrusted LLM output.
        result = str(eval(code, {"df": df, "pd": pd}))  # noqa: S307
    except Exception as e:
        result = f"ERROR: {e}"
    print(f"{q[:53]:<55} {code[:68]:<70} {result}")

---

## Part 4 — Self-Correction Loop

---

When `eval()` raises an exception, we feed the error message back to the LLM and ask it to fix the expression. This is the self-correction loop.

### When Does Code Generation Fail?

Common failure modes:
- **Wrong column name** — LLM guesses `revenue` but the column is `units_sold`
- **Type mismatch** — LLM tries `df["price"].sum()` on a string column
- **Method doesn't exist** — LLM calls `df.argmax()` instead of `df.idxmax()`
- **Complex aggregation** — LLM generates valid Python but wrong logic (e.g., max price vs max revenue)

Feeding the exact error message lets the LLM understand exactly what went wrong and fix it.

In [ ]:
# 3 retries catches most one-off pandas syntax mistakes; more iterations
# rarely help — if the model is confused it keeps making the same error.
MAX_ITERATIONS = 3


def execute_with_retry(question: str, verbose: bool = True) -> dict:
    error = ""
    for iteration in range(1, MAX_ITERATIONS + 1):
        code = text_to_pandas(question, error=error)
        if verbose:
            print(f"  [iter {iteration}] code: {code}")
        try:
            result = str(eval(code, {"df": df, "pd": pd}))  # noqa: S307
            return {"question": question, "code": code, "result": result, "iterations": iteration}
        except Exception as exc:
            error = str(exc)
            if verbose:
                print(f"  [iter {iteration}] error: {error}")
    return {"question": question, "code": code, "result": "", "error": error, "iterations": MAX_ITERATIONS}


# Demonstrate with a deliberately tricky question
tricky = "What is the total revenue for each category?"
print(f"Question: '{tricky}'")
out = execute_with_retry(tricky, verbose=True)
print(f"Result: {out.get('result', out.get('error', 'max retries exceeded'))}")

---

## Part 5 — Full Pipeline with LangGraph

---

The graph has a **conditional edge** out of `execute_query`: if there was an error and we haven't hit `MAX_ITERATIONS`, loop back to `write_query` with the error attached to state. Otherwise, end.

```
START
  │
  ▼
load_table   ─── parse CSV, build schema + sample context
  │
  ▼
write_query  ─── LLM generates pandas expression (re-prompted with error if any)
  │
  ▼
execute_query ── eval() the expression
  │
  ├─ error AND iterations < MAX  ──────────→  write_query (retry)
  │
  └─ success OR max iterations   ──────────→  END
```

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


# TypedDict gives LangGraph typed access to state keys — every node returns
# a dict with only the keys it updates; other keys are merged in automatically.
class TabularState(TypedDict):
    question:   str
    schema:     str
    query_code: str
    result:     str
    error:      str
    iterations: int


def load_table_node(state: TabularState) -> dict:
    return {"schema": schema, "error": "", "iterations": 0}


def write_query_node(state: TabularState) -> dict:
    code = text_to_pandas(state["question"], error=state.get("error", ""))
    return {"query_code": code, "iterations": state["iterations"] + 1}


def execute_query_node(state: TabularState) -> dict:
    try:
        result = eval(state["query_code"], {"df": df, "pd": pd})  # noqa: S307
        return {"result": str(result), "error": ""}
    except Exception as exc:
        return {"result": "", "error": str(exc)}


def route(state: TabularState) -> Literal["write_query", "__end__"]:
    if state["error"] and state["iterations"] < MAX_ITERATIONS:
        return "write_query"
    return END


graph = StateGraph(TabularState)
graph.add_node("load_table", load_table_node)
graph.add_node("write_query", write_query_node)
graph.add_node("execute_query", execute_query_node)
graph.add_edge(START, "load_table")
graph.add_edge("load_table", "write_query")
graph.add_edge("write_query", "execute_query")
# add_conditional_edges: route() return value maps to a node name or END —
# this is how the self-correction loop is wired without explicit looping code.
graph.add_conditional_edges("execute_query", route)
# compile() locks the topology and returns an invocable app;
# pass checkpointer=MemorySaver() here to persist state across calls.
app = graph.compile()

print("Pipeline compiled: load_table → write_query → execute_query → [conditional retry]")

In [ ]:
# ─── Run all three questions through the full pipeline ───────────────────────

for question in SAMPLE_QUESTIONS:
    print(f"{'=' * 65}")
    result: TabularState = app.invoke({
        "question":   question,
        "schema":     "",
        "query_code": "",
        "result":     "",
        "error":      "",
        "iterations": 0,
    })
    print(f"Question   : {result['question']}")
    print(f"Code       : {result['query_code']}")
    print(f"Iterations : {result['iterations']}")
    if result["result"]:
        print(f"Answer     : {result['result']}")
    else:
        print(f"Error      : {result['error']} (max retries reached)")
    print()

---

### Exercise 1 — Verify the Answers Manually

**Goal:** Write the pandas expressions yourself (without the LLM) to verify the pipeline's answers are correct.

1. Highest-revenue product: `price × units_sold` for each row → find the max
2. Average price of Electronics
3. Total units sold in North region

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────

# Q1: Which product has the highest total revenue?
# TODO: df.assign(...) or df["price"] * df["units_sold"] → idxmax
ans1 = None  # replace with your expression

# Q2: Average price of Electronics
ans2 = None  # replace with your expression

# Q3: Total units sold in the North region
ans3 = None  # replace with your expression

print(f"Q1 (highest revenue product): {ans1}")
print(f"Q2 (avg Electronics price)  : {ans2}")
print(f"Q3 (total North units)       : {ans3}")

### Exercise 2 — Add Your Own Question

**Goal:** Run the full pipeline with a question that requires a *multi-step* computation:

```
YOUR_QUESTION = "What is the revenue share (%) of Furniture vs total revenue?"
```

1. Feed it to `app.invoke()`
2. Print the generated code and the answer
3. Verify the answer manually with a direct pandas expression

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────

YOUR_QUESTION = "What is the revenue share (%) of Furniture vs total revenue?"

# TODO: invoke app and print code, iterations, result
# TODO: write the direct pandas expression to verify
pass

---

## What's Next?

You now understand tabular RAG: replace embedding retrieval with code generation + self-correction for structured data.

### Combine with prose RAG
- **Hybrid approach:** Route questions to tabular RAG (for computation) or embedding RAG (for prose) based on question type — a semantic router (Example 64) can make this routing decision automatically
- **Example 14 — SQL Agent** ([`../14-sql-agent/README.md`](../14-sql-agent/README.md)): the same idea but with a real database — the LLM writes SQL instead of pandas, and a real DB engine executes it

### Scale tabular RAG
- For large tables (millions of rows), `eval()` on a DataFrame is fast — the main cost is the LLM call
- For very complex schemas, add a `describe_table` step that includes `df.describe()` and `df.dtypes` in the prompt
- For multiple tables, add a `select_table` node that routes to the right DataFrame before generating code

### Security note
- `eval()` is used here for teaching purposes on trusted, LLM-generated code
- In production, use `pandas-eval` sandboxes or restrict `eval()` to a whitelist of safe pandas operations

---

*Workshop complete. You've built all 5 advanced RAG notebooks: reranking, HyDE, contextual compression, step-back prompting, and tabular RAG.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself.

In [ ]:
# Exercise 1 — Manual verification

# Q1: Product with highest revenue
revenue = df["price"] * df["units_sold"]
ans1 = df.loc[revenue.idxmax(), "product"]

# Q2: Average price of Electronics
ans2 = df[df["category"] == "Electronics"]["price"].mean()

# Q3: Total units sold in North region
ans3 = df[df["region"] == "North"]["units_sold"].sum()

print(f"Q1 (highest revenue product): {ans1}")
print(f"Q2 (avg Electronics price)  : {ans2:.2f}")
print(f"Q3 (total North units)       : {ans3}")

# Show the full revenue table for transparency
print()
print("Full revenue breakdown:")
print(
    df.assign(revenue=revenue)
      .sort_values("revenue", ascending=False)
      [["product", "price", "units_sold", "revenue"]]
      .to_string(index=False)
)

In [ ]:
# Exercise 2 — Revenue share of Furniture

result = app.invoke({
    "question":   YOUR_QUESTION,
    "schema":     "",
    "query_code": "",
    "result":     "",
    "error":      "",
    "iterations": 0,
})
print(f"LLM code  : {result['query_code']}")
print(f"Iterations: {result['iterations']}")
print(f"Answer    : {result['result']}")

# Manual verification
total_rev = (df["price"] * df["units_sold"]).sum()
furniture_rev = (df[df["category"] == "Furniture"]["price"] * df[df["category"] == "Furniture"]["units_sold"]).sum()
pct = furniture_rev / total_rev * 100
print(f"Manual verification: Furniture revenue share = {pct:.2f}%")